# Download and create data

/data folder contain a already merged and preprocessed data.
But if you want to go to the original sources and how is it filtered out or merged, you can rely on it or use this notebook to understand how and why.
This is a needed step if you want to run this framework with other dataset.

- mergedcounts_generation.py
- metadata_generation.py



## Running against your own sandbox copy

The cells below never touch `data/` directly. They:

1. Copy just the inputs these two scripts read from `data/` (`metadata.csv`, `mergedcounts.csv`, and the `ucam_sanyal/counts_matrix.csv` duplicate used for cross-checking) into a **fresh** `sandbox/data_dev/` folder -- wiping and recreating it every time the first cell runs, so you always start from a clean copy.
2. Point `mergedcounts_generation.py` / `metadata_generation.py` at that copy via the `MASLD_DATA_DIR` environment variable (both scripts fall back to the real `data/` when it's unset) and import them as modules.
3. Call their `main()` functions against `sandbox/data_dev/` -- any `--write` output lands there, never in `data/`.

`sandbox/data_dev/` is already gitignored, so nothing produced here is ever committed. Re-run the first cell any time you want to start over.


In [1]:
import importlib
import os
import shutil
import sys
from pathlib import Path

# Locate the repo root regardless of where Jupyter was launched from
# (e.g. `pixi run jupyter lab notebooks/` from sandbox/, per sandbox/CLAUDE_NOTES.md).
REPO_ROOT = Path.cwd().resolve()
for _ in range(6):
    if (REPO_ROOT / "src").is_dir() and (REPO_ROOT / "data").is_dir():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError(
        "Could not locate the repo root (a directory containing both src/ and "
        "data/) above the current working directory."
    )

DATA_DIR = REPO_ROOT / "data"
DATA_DEV_DIR = REPO_ROOT / "sandbox" / "data_dev"
SCRIPTS_DIR = REPO_ROOT / "src" / "00.download_and_merge_data"

# Files mergedcounts_generation.py / metadata_generation.py read from DATA_DIR.
# Add to this list as later sandbox steps need more inputs copied from data/.
SEED_FILES = [
    "metadata.csv",
    "mergedcounts.csv",
    "ucam_sanyal/counts_matrix.csv",  # known-duplicate cross-check
]


def fresh_data_dev() -> None:
    """Wipe and recreate sandbox/data_dev/, seeded with copies of the source
    files these two scripts need from data/. Safe to re-run any time you
    want a clean copy -- data/ itself is only ever read, never modified."""
    if DATA_DEV_DIR.exists():
        shutil.rmtree(DATA_DEV_DIR)
    DATA_DEV_DIR.mkdir(parents=True)
    for rel_path in SEED_FILES:
        src = DATA_DIR / rel_path
        dest = DATA_DEV_DIR / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
    print(f"[OK] Seeded {DATA_DEV_DIR} with {len(SEED_FILES)} file(s) copied from {DATA_DIR}.")


fresh_data_dev()

# Point the scripts at sandbox/data_dev/ instead of data/, then (re)import them
# so their module-level path constants pick up the new MASLD_DATA_DIR.
os.environ["MASLD_DATA_DIR"] = str(DATA_DEV_DIR)
sys.path.insert(0, str(SCRIPTS_DIR))

import mergedcounts_generation
import metadata_generation

importlib.reload(mergedcounts_generation)
importlib.reload(metadata_generation)

print(f"[OK] mergedcounts_generation.DATA_DIR = {mergedcounts_generation.DATA_DIR}")
print(f"[OK] metadata_generation.DATA_DIR     = {metadata_generation.DATA_DIR}")


[OK] Seeded /home/koryclick/Documents/git_projects/MASLD_ContinuousTrajectoryApproach/sandbox/data_dev with 3 file(s) copied from /home/koryclick/Documents/git_projects/MASLD_ContinuousTrajectoryApproach/data.
[OK] mergedcounts_generation.DATA_DIR = /home/koryclick/Documents/git_projects/MASLD_ContinuousTrajectoryApproach/sandbox/data_dev
[OK] metadata_generation.DATA_DIR     = /home/koryclick/Documents/git_projects/MASLD_ContinuousTrajectoryApproach/sandbox/data_dev


In [2]:
# Validates the seeded sandbox/data_dev/mergedcounts.csv against
# sandbox/data_dev/metadata.csv, then rewrites it with sample-name columns.
# data/mergedcounts.csv itself is never touched (MASLD_DATA_DIR redirected
# DATA_DIR to sandbox/data_dev/ above).
mergedcounts_generation.main(write=True)


[fallback] Per-cohort raw HTSeq matrices are not present in this repo (sandbox/data_dev/raw/ucam_htseq_counts.tsv, sandbox/data_dev/raw/vcu_htseq_counts.tsv not found). Falling back to validating the existing, already-merged mergedcounts.csv instead of regenerating it from scratch.
[filter] Low-count filter (mean count >= 1.0): kept 17090 / 17090 genes (dropped 0).
[OK] Committed matrix already satisfies mean count >= 1.0 (filter is a no-op).
[OK] Matches the known duplicate copy at sandbox/data_dev/ucam_sanyal/counts_matrix.csv.
[OK] 136 samples, 17090 Ensembl genes; dataset composition {'SANYAL': 78, 'UCAM': 58}; outlier 'Sample 5' present (excluded later, in 'Batch correction UCAM-VCU (...).R', not here).
[write] sandbox/data_dev/mergedcounts.csv regenerated with sample-name columns.


In [ ]:
# Cross-checks sandbox/data_dev/metadata.csv against the public GEO/ArrayExpress
# records (network access required). Never writes -- see the script's
# docstring for why metadata.csv can't be fully regenerated from scratch.
metadata_generation.main()


`sandbox/data_dev/` now holds an independently validated/regenerated copy of `mergedcounts.csv` alongside the seeded `metadata.csv` -- inspect it, diff it against `data/`, or just re-run `fresh_data_dev()` above to blow it away and start over.


## Optional: true from-scratch regeneration from raw FASTQ (`regenerate_from_raw_fastq.py`)

**This is a separate, independent script -- not a later stage of the two above.** It produces the *complete* final matrix by itself; you do not run `mergedcounts_generation.py` afterward to "finish" it.

Why a third script, when `mergedcounts_generation.py` already claims to "merge from raw counts"? Because that script's real merge path only fires if you already have the two cohorts' separate per-sample HTSeq count matrices sitting locally -- nobody does, in this repo, so in practice it always falls back to validating/relabelling the already-committed `data/mergedcounts.csv`. It never touches actual sequencing reads.

`regenerate_from_raw_fastq.py` is the one that does: it downloads the real raw paired-end FASTQ for both cohorts (UCAM via ArrayExpress E-MTAB-9815/ENA, VCU/Sanyal via GEO GSE130970 -> SRA SRP197353), runs each sample through FastQC -> HISAT2 (GRCh38) -> HTSeq itself, then assembles, inner-joins, and filters the two cohorts on its own -- the same steps `mergedcounts_generation.py` would run *if* it had the raw material. It writes to a **different file**, `mergedcounts_from_raw_fastq.csv` (never `mergedcounts.csv`), specifically so it can't silently overwrite or be confused with the committed one -- treat it as an independent reproduction to diff against, not a drop-in replacement.

**When to actually reach for it:** essentially never for routine work. Use `mergedcounts_generation.py` above for everyday reproducibility -- it reproduces the committed table exactly, in seconds, no downloads. Use `regenerate_from_raw_fastq.py` only if you specifically want to verify the paper's pipeline end-to-end starting from the public raw reads, or need to run a *different* raw-FASTQ cohort through the same steps. Real cost: ~338 GB of downloads and, per an informal (unbenchmarked-on-any-specific-machine) estimate, **4-12+ days** of alignment compute on a machine with ~8 GB RAM -- see the script's own module docstring for the full breakdown, including two undocumented-in-the-paper assumptions it has to make (Ensembl annotation release, library strandedness) that mean it is not expected to be byte-identical to `data/mergedcounts.csv`.

The cell below only runs `--dry-run` -- it fetches the real sample manifest (network reads only) and prints sizes/read counts, no downloads, no bioinformatics tools required. `--benchmark` (times 1 sample, extrapolates a real estimate for your machine) and `--full` (the actual multi-day run) are deliberately not run from this notebook -- see the commands underneath.


In [3]:
import subprocess

# Cheap sanity check only: fetches the real sample manifest from ArrayExpress/
# ENA + GEO/SRA and prints sizes/read counts. No downloads, no tools required.
subprocess.run(
    [sys.executable, str(SCRIPTS_DIR / "regenerate_from_raw_fastq.py"), "--dry-run"],
    env={**os.environ, "MASLD_DATA_DIR": str(DATA_DEV_DIR)},
    check=True,
)


[manifest] Fetching UCAM sample manifest from ArrayExpress/ENA ...
[manifest] UCAM: 58 samples.
[manifest] Fetching VCU/Sanyal sample manifest from GEO/ENA ...
[manifest] VCU/Sanyal: 78 samples.

[dry-run] 136 samples total, 338.3 GB, 4,875,297,674 reads.
  UCAM: 58 samples, 21.6 GB, 561,926,617 reads
  SANYAL: 78 samples, 316.6 GB, 4,313,371,057 reads
No downloads performed, no tools required for this check.


CompletedProcess(args=['/home/koryclick/Documents/git_projects/MASLD_ContinuousTrajectoryApproach/sandbox/.pixi/envs/default/bin/python', '/home/koryclick/Documents/git_projects/MASLD_ContinuousTrajectoryApproach/src/00.download_and_merge_data/regenerate_from_raw_fastq.py', '--dry-run'], returncode=0)

---
> **⚠️ WARNING — Real multi-day run (~338 GB)**
>
> The `--full` flag launches the actual end-to-end pipeline:
> - **~338 GB** of FASTQ downloads (UCAM: 21.6 GB · VCU/Sanyal: 316.6 GB)
> - **136 samples** through FastQC → HISAT2 (GRCh38) → HTSeq-count
> - Rough estimate: **4–12+ days** on a machine with ~8 GB RAM (single thread)
> - Output written to **`mergedcounts_from_raw_fastq.csv`** — never overwrites `data/mergedcounts.csv`
>
> **Run the pre-flight checks below before launching.**

In [ ]:
import multiprocessing
import shutil
import subprocess

print("=" * 62)
print("  PRE-FLIGHT CHECKS — full ~338 GB regeneration run")
print("=" * 62)

# ── 1. System RAM ─────────────────────────────────────────────────────────────
with open("/proc/meminfo") as _f:
    _meminfo = {line.split(":")[0]: line.split(":", 1)[1].strip() for line in _f}
total_ram_gb = int(_meminfo["MemTotal"].split()[0])    / 1024**2
avail_ram_gb = int(_meminfo["MemAvailable"].split()[0]) / 1024**2
print(f"\n[RAM]  System  : {total_ram_gb:.1f} GB total | {avail_ram_gb:.1f} GB available")

# ── 2. CPU cores ──────────────────────────────────────────────────────────────
cpu_count = multiprocessing.cpu_count()
print(f"[CPU]  Cores   : {cpu_count} logical")

# ── 3. Disk space (where downloads will land) ─────────────────────────────────
_usage = shutil.disk_usage(DATA_DEV_DIR)
free_disk_gb  = _usage.free  / 1024**3
total_disk_gb = _usage.total / 1024**3
disk_ok = free_disk_gb >= 400  # 338 GB data + headroom for BAMs/intermediates
print(f"[DISK] Free    : {free_disk_gb:.0f} GB / {total_disk_gb:.0f} GB  "
      f"({'OK' if disk_ok else '!! INSUFFICIENT — need ≥400 GB free'})")

# ── 4. NVIDIA GPU VRAM ────────────────────────────────────────────────────────
# HISAT2 runs on CPU; GPU is not used by the pipeline — reported for awareness.
_nsmi = shutil.which("nvidia-smi")
if _nsmi:
    _r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    if _r.returncode == 0 and _r.stdout.strip():
        for _i, _line in enumerate(_r.stdout.strip().splitlines()):
            _name, _tot, _free = [x.strip() for x in _line.split(",")]
            try:
                _tot_gb  = int(_tot)  / 1024
                _free_gb = int(_free) / 1024
                print(f"[GPU {_i}] {_name}: {_tot_gb:.1f} GB VRAM total | "
                      f"{_free_gb:.1f} GB free  (pipeline uses CPU, not GPU)")
            except ValueError:
                # Driver reports [N/A] for memory (common on some kernels/VMs)
                print(f"[GPU {_i}] {_name}: VRAM info unavailable (driver returned '{_tot}') "
                      f"— pipeline uses CPU, not GPU")
    else:
        print("[GPU]  nvidia-smi found but returned no data")
else:
    print("[GPU]  No NVIDIA GPU detected (not required — pipeline is CPU-only)")

# ── 5. Required bioinformatics tools ──────────────────────────────────────────
REQUIRED_TOOLS = ["fastqc", "hisat2", "samtools", "htseq-count"]
print()
_missing = []
for _tool in REQUIRED_TOOLS:
    _path = shutil.which(_tool)
    if _path:
        try:
            _v = subprocess.run([_tool, "--version"], capture_output=True, text=True)
            _ver = (_v.stdout or _v.stderr).splitlines()[0][:55]
        except Exception:
            _ver = "(version unknown)"
        print(f"  [OK] {_tool:15s}  {_ver}")
    else:
        print(f"  [!!] {_tool:15s}  MISSING")
        _missing.append(_tool)

if _missing:
    print(f"\n  Install missing tools (from sandbox/):")
    print(f"  pixi add -c bioconda -c conda-forge {' '.join(_missing)}")

# ── 6. Runtime estimate ───────────────────────────────────────────────────────
# Baseline: script docstring — ~4–12 days (midpoint 8 d) sequential on 8 GB RAM.
# HISAT2 GRCh38 index ≈ 8 GB; if RAM < 8 GB the index must be paged (~10× penalty).
# Parallel samples share the loaded index; each extra sample adds ~0.5 GB overhead.
print()
_BASE_DAYS   = 8.0
_BASE_RAM_GB = 8.0   # HISAT2 GRCh38 index size

if total_ram_gb < _BASE_RAM_GB:
    _ram_factor    = (_BASE_RAM_GB / total_ram_gb) * 10.0
    _effective_par = 1
    _ram_note      = f"index must page to disk ({total_ram_gb:.1f} GB < {_BASE_RAM_GB:.0f} GB needed) — severe penalty"
else:
    _ram_factor      = 1.0
    _max_par_by_ram  = max(1, int((total_ram_gb - _BASE_RAM_GB) / 0.5) + 1)
    _effective_par   = min(cpu_count, 8, _max_par_by_ram)
    _ram_note        = f"index fits in RAM; up to {_effective_par} samples in parallel"

_estimated_days = (_BASE_DAYS * _ram_factor) / _effective_par

print(f"[TIME] Estimate (baseline ~{_BASE_DAYS:.0f} d sequential on {_BASE_RAM_GB:.0f} GB RAM):")
print(f"       RAM       : {_ram_note}")
print(f"       Parallel  : --parallel {_effective_par}  ({cpu_count} cores, RAM-limited to {_effective_par})")
print(f"       Estimated : ~{_estimated_days:.1f} days on this machine")
print(f"       (Run --benchmark on 1 sample for a machine-specific measurement)")

# ── 7. Summary ────────────────────────────────────────────────────────────────
print()
print("─" * 62)
_all_ok = not _missing and disk_ok
if _all_ok:
    print(f"[OK] All checks passed. Suggested command (from src/00.download_and_merge_data/):")
    print(f"     MASLD_DATA_DIR=../../sandbox/data_dev \\")
    print(f"         python regenerate_from_raw_fastq.py --full --parallel {_effective_par}")
else:
    if _missing:
        print(f"[!!] Missing tools : {', '.join(_missing)}")
    if not disk_ok:
        print(f"[!!] Disk space    : {free_disk_gb:.0f} GB free — need ≥400 GB")
    print("     Resolve the issues above before launching --full.")
print("─" * 62)


To actually benchmark or run the real regeneration (outside this notebook, from `src/00.download_and_merge_data/`, ideally on a machine with well over 8 GB RAM):

```bash
# Times 1 sample end-to-end, extrapolates a real estimate for this machine:
MASLD_DATA_DIR=../../sandbox/data_dev python regenerate_from_raw_fastq.py --benchmark

# The real, multi-day, ~338 GB run:
MASLD_DATA_DIR=../../sandbox/data_dev python regenerate_from_raw_fastq.py --full --parallel 4
```

Both require `fastqc`, `hisat2`, `samtools`, and `htseq-count` on `PATH` (e.g. `pixi add -c bioconda -c conda-forge fastqc hisat2 samtools htseq`) -- neither is installed in this sandbox env yet.


---
> **⚠️ WARNING — This cell launches the real multi-day run**
>
> Running the cell below will start the full `--full` pipeline (~338 GB downloads, ~1 day on this machine with `--parallel 4`). Before you do:
>
> **Prefer the terminal over this notebook.**
> A Jupyter kernel is not designed for multi-day jobs — it will be killed by a browser disconnect, an idle timeout, a kernel restart, or an accidental notebook closure, discarding all progress. Running from the terminal is strongly recommended instead:
>
> ```bash
> # Option 1 — tmux (survives terminal closure, recommended):
> tmux new -s masld_full
> cd src/00.download_and_merge_data
> MASLD_DATA_DIR=../../sandbox/data_dev python regenerate_from_raw_fastq.py --full --parallel 4
> # detach with Ctrl-B D; reattach later with: tmux attach -t masld_full
>
> # Option 2 — nohup (fire-and-forget, logs to nohup.out):
> cd src/00.download_and_merge_data
> nohup env MASLD_DATA_DIR=../../sandbox/data_dev \
>     python regenerate_from_raw_fastq.py --full --parallel 4 &
> ```
>
> Only run the cell below if you are intentionally running inside the notebook and accept the risk of interruption.

In [ ]:
import subprocess

# Full end-to-end regeneration from raw FASTQ — see warning above before running.
# Output: mergedcounts_from_raw_fastq.csv (never overwrites data/mergedcounts.csv).
subprocess.run(
    [
        sys.executable,
        str(SCRIPTS_DIR / "regenerate_from_raw_fastq.py"),
        "--full",
        "--parallel", "4",
    ],
    env={**os.environ, "MASLD_DATA_DIR": str(DATA_DEV_DIR)},
    check=True,
)
